# MVFoul — Three Baselines Side-by-Side

Each approach uses identical train/valid splits, class weights, and metrics.

| | Approach | Architecture |
|---|---|---|
| **A** | MViT video baseline | MViTv2-S pretrained + max-pool across views |
| **B** | Pose + BiLSTM | MediaPipe 33-kp → Bidirectional LSTM |
| **C** | ST-GCN (novel) | Velocity-augmented skeleton graph + ST-GCN |

`DEBUG_MODE = True` by default — runs on 60 actions (~10 min on T4). Set False for full run.
No clips? Cell 4 detects this and runs in synthetic mode — all architectures still execute.


## Cell 1 — Install

In [ ]:
!pip install pytorchvideo --quiet
!pip install mediapipe --quiet
!pip install ultralytics --quiet
!pip install SoccerNet --quiet
!pip install av --quiet
!pip install scikit-learn --quiet
print('Done')


## Cell 2 — Imports and config

In [ ]:
import json, os, zipfile, random, time, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix
warnings.filterwarnings('ignore')

DEBUG_MODE  = True
DEBUG_N     = 60
SEED        = 42
DATA_DIR    = Path('/content/mvfoul_data')
MVFOUL_DIR  = DATA_DIR / 'mvfouls'
CACHE_DIR   = Path('/content/feature_cache')
CKPT_DIR    = Path('/content/checkpoints')
RESULTS_DIR = Path('/content/results')
for d in [CACHE_DIR, CKPT_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {name}  VRAM: {mem:.1f} GB')

ACTION_CLASSES = ['Tackling','Standing tackling','High leg','Holding',
                  'Pushing','Elbowing','Challenge','Dive']
CLASS2IDX = {c: i for i, c in enumerate(ACTION_CLASSES)}
N_CLASSES  = len(ACTION_CLASSES)
DIVE_IDX   = CLASS2IDX['Dive']
SEVERITY_MAP = {'1.0':'No card','3.0':'Yellow card','5.0':'Red card',
                '2.0':'No card/Yellow card','4.0':'Yellow card/Red card','':'Unknown'}
print(f'Dive index: {DIVE_IDX}  N_CLASSES: {N_CLASSES}')


## Cell 3 — Load annotations

In [ ]:
def load_split_df(json_path, split_name):
    with open(json_path) as f:
        raw = json.load(f)
    actions = raw.get('Actions', raw.get('Set', {}))
    rows = []
    for action_id, data in actions.items():
        clips  = data.get('clips', [])
        action = data.get('Action class', '').strip()
        if action not in CLASS2IDX:
            action = 'Dont know'
        rows.append({
            'action_id'  : action_id,
            'split'      : split_name,
            'label'      : CLASS2IDX.get(action, -1),
            'action_name': action,
            'is_dive'    : action == 'Dive',
            'clip_paths' : [c.get('path','') for c in clips],
            'Severity'   : SEVERITY_MAP.get(str(data.get('Severity','')).strip(), 'Unknown'),
        })
    return pd.DataFrame(rows)

split_dfs = {}
for json_path in sorted(MVFOUL_DIR.rglob('*.json')):
    p = str(json_path).lower()
    for s in ['train','valid','test']:
        if s in p and s not in split_dfs:
            split_dfs[s] = load_split_df(json_path, s)
            break

if not split_dfs:
    raise RuntimeError('No JSONs found. Run notebook 1 first.')

for s in split_dfs:
    before = len(split_dfs[s])
    split_dfs[s] = split_dfs[s][split_dfs[s]['label'] >= 0].reset_index(drop=True)
    print(f'{s}: {len(split_dfs[s])} actions (dropped {before-len(split_dfs[s])} unknown)')

if DEBUG_MODE:
    for s in split_dfs:
        split_dfs[s] = (
            split_dfs[s]
            .groupby('label', group_keys=False)
            .apply(lambda x: x.sample(min(len(x), max(1, DEBUG_N // N_CLASSES)),
                                       random_state=SEED))
            .reset_index(drop=True)
        )
    print('DEBUG subsets:', {s: len(df) for s, df in split_dfs.items()})

train_df = split_dfs.get('train', pd.DataFrame())
valid_df  = split_dfs.get('valid', pd.DataFrame())
print(f'Train: {len(train_df)}  Valid: {len(valid_df)}')


## Cell 4 — Diagnostic: what was downloaded?

In [ ]:
print('='*60)
print('DOWNLOAD DIAGNOSTIC')
print('='*60)

zip_files = sorted(MVFOUL_DIR.glob('*.zip'))
for z in zip_files:
    mb   = z.stat().st_size / 1e6
    kind = 'annotations only (<10MB)' if mb < 10 else 'includes video clips'
    print(f'  {z.name:30s} {mb:8.1f} MB  [{kind}]')

VIDEO_EXTS  = {'.mp4','.avi','.mkv','.mov'}
video_files = [p for p in MVFOUL_DIR.rglob('*') if p.suffix.lower() in VIDEO_EXTS]
print(f'JSON files : {len(list(MVFOUL_DIR.rglob("*.json")))}')
print(f'Video files: {len(video_files)}')

HAS_VIDEOS   = len(video_files) > 0
video_lookup = defaultdict(list)

if HAS_VIDEOS:
    for vp in video_files:
        video_lookup[vp.parent.name].append(vp)
    ann_ids = set(train_df['action_id']) | set(valid_df['action_id'])
    covered = ann_ids & set(video_lookup.keys())
    print(f'Coverage: {len(covered)}/{len(ann_ids)} annotated actions have clips')
    print('STATUS: VIDEO CLIPS FOUND')
else:
    print('STATUS: ANNOTATIONS ONLY -- SYNTHETIC MODE')
    print('  Re-run notebook 1 Cell 3 with your SoccerNet password to get clips.')

print(f'HAS_VIDEOS = {HAS_VIDEOS}')


## Cell 5 — Shared utilities

In [ ]:
class_counts = train_df['label'].value_counts().sort_index()
counts_arr   = np.array([class_counts.get(i,1) for i in range(N_CLASSES)], dtype=float)
inv_freq     = 1.0 / counts_arr
weights      = torch.tensor(inv_freq / inv_freq.mean(), dtype=torch.float32).to(DEVICE)

print('Class weights (mean=1):')
for i, (cls, w) in enumerate(zip(ACTION_CLASSES, weights.cpu().numpy())):
    flag = '  <- Dive' if cls == 'Dive' else ''
    print(f'  [{i}] {cls:<22s} {w:.3f}{flag}')


def compute_metrics(y_true, y_pred):
    ba  = balanced_accuracy_score(y_true, y_pred)
    mac = f1_score(y_true, y_pred, average='macro',
                   labels=list(range(N_CLASSES)), zero_division=0)
    df1 = f1_score(y_true, y_pred, labels=[DIVE_IDX], average='macro', zero_division=0)
    cm  = confusion_matrix(y_true, y_pred, labels=list(range(N_CLASSES)))
    return {'balanced_acc': round(ba,4), 'dive_f1': round(df1,4),
            'macro_f1': round(mac,4), 'conf_matrix': cm}


def plot_confusion_matrix(cm, title, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(8,6))
    labels  = [c[:10] for c in ACTION_CLASSES]
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=labels, yticklabels=labels,
                linewidths=0.3, ax=ax, cbar=False)
    ax.set_title(title)
    ax.set_ylabel('True'); ax.set_xlabel('Predicted')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=8)


def train_one_epoch(model, loader, optimizer, scaler, criterion):
    model.train()
    total_loss, preds, trues = 0.0, [], []
    for x, y in loader:
        x = x.to(DEVICE); y = y.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            logits = model(x)
            loss   = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * y.size(0)
        preds.extend(logits.detach().argmax(1).cpu().numpy())
        trues.extend(y.cpu().numpy())
    return total_loss / len(trues), balanced_accuracy_score(trues, preds)


@torch.no_grad()
def run_eval(model, loader):
    model.eval()
    preds, trues = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        with autocast():
            preds.extend(model(x).argmax(1).cpu().numpy())
        trues.extend(y.numpy())
    return compute_metrics(trues, preds)


def train_approach(name, model, train_dl, valid_dl,
                   n_epochs=15, lr=1e-4, patience=5, ckpt_name=None):
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    scaler    = GradScaler()
    ckpt_path = CKPT_DIR / f'{ckpt_name or name}.pt'
    best_ba, patience_ctr = 0.0, 0
    history = []
    print(f'Training {name} | {n_epochs} epochs | lr={lr}')
    for ep in range(1, n_epochs+1):
        t0 = time.time()
        tr_loss, tr_ba = train_one_epoch(model, train_dl, optimizer, scaler, criterion)
        vm = run_eval(model, valid_dl)
        scheduler.step()
        history.append({'epoch':ep,'tr_loss':tr_loss,'tr_ba':tr_ba,
                        'val_ba':vm['balanced_acc'],'val_dive_f1':vm['dive_f1']})
        flag = ''
        if vm['balanced_acc'] > best_ba:
            best_ba = vm['balanced_acc']
            torch.save(model.state_dict(), ckpt_path)
            patience_ctr = 0; flag = ' [saved]'
        else:
            patience_ctr += 1
        elapsed = time.time() - t0
        print(f'  ep {ep:02d}/{n_epochs}  loss={tr_loss:.3f}  tr_ba={tr_ba:.3f}'
              f'  val_ba={vm["balanced_acc"]:.3f}  dive_f1={vm["dive_f1"]:.3f}'
              f'  ({elapsed:.1f}s){flag}')
        if patience_ctr >= patience:
            print(f'  Early stop at epoch {ep}'); break
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    final = run_eval(model, valid_dl)
    print(f'Best val_ba={best_ba:.4f}  dive_f1={final["dive_f1"]:.4f}')
    return final, pd.DataFrame(history)

print('Shared utilities ready')


---
## Approach A — MViT Video Baseline
Replicates VARS (Held et al. 2023). Two-phase training: head only first, then full fine-tune.


### Cell A1 — Video dataset + model

In [ ]:
def load_clip_frames(video_path, n_frames=16, size=224):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened(): return None
    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, max(total-1,0), n_frames, dtype=int)
    frames  = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok: frame = np.zeros((size,size,3), dtype=np.uint8)
        frame = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (size,size))
        frames.append(frame)
    cap.release()
    t    = torch.from_numpy(np.stack(frames)).float() / 255.0
    mean = torch.tensor([0.45,0.45,0.45]).view(1,1,1,3)
    std  = torch.tensor([0.225,0.225,0.225]).view(1,1,1,3)
    return ((t-mean)/std).permute(3,0,1,2)  # (3,T,H,W)


class MVFoulVideoDataset(Dataset):
    def __init__(self, df, video_lookup, n_frames=16, size=224):
        self.df=df.reset_index(drop=True); self.vl=video_lookup
        self.n_frames=n_frames; self.size=size
        self.synthetic=len(video_lookup)==0
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]; label=int(row['label'])
        if self.synthetic:
            return torch.randn(3,self.n_frames,self.size,self.size), label
        clips   = self.vl.get(row['action_id'],[])
        replays = [c for c in clips if 'replay' in str(c).lower()]
        for vp in (replays or clips):
            t = load_clip_frames(vp, self.n_frames, self.size)
            if t is not None: return t, label
        return torch.randn(3,self.n_frames,self.size,self.size), label


class ApproachA_MViT(nn.Module):
    def __init__(self, n_classes, freeze_backbone=True):
        super().__init__()
        self.backbone = torch.hub.load(
            'facebookresearch/pytorchvideo', 'mvit_v2_s', pretrained=True
        )
        in_feat = self.backbone.head.proj.in_features
        self.backbone.head.proj = nn.Linear(in_feat, n_classes)
        if freeze_backbone:
            for nm, p in self.backbone.named_parameters():
                if 'head' not in nm: p.requires_grad = False
    def unfreeze(self):
        for p in self.backbone.parameters(): p.requires_grad = True
    def forward(self, x): return self.backbone(x)

print('Approach A defined')


### Cell A2 — Train Approach A

In [ ]:
if not HAS_VIDEOS: print('SYNTHETIC MODE')

train_dl_A = DataLoader(MVFoulVideoDataset(train_df, video_lookup),
                        batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
valid_dl_A = DataLoader(MVFoulVideoDataset(valid_df, video_lookup),
                        batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

model_A = ApproachA_MViT(n_classes=N_CLASSES, freeze_backbone=True).to(DEVICE)
trainable = sum(p.numel() for p in model_A.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_A.parameters())
print(f'Approach A | trainable: {trainable:,} / {total:,}')

# Phase 1: frozen backbone
metrics_A, hist_A1 = train_approach('A-frozen', model_A, train_dl_A, valid_dl_A,
    n_epochs=5, lr=3e-4, patience=3, ckpt_name='approach_A')

# Phase 2: full fine-tune
model_A.unfreeze()
metrics_A, hist_A2 = train_approach('A-full', model_A, train_dl_A, valid_dl_A,
    n_epochs=10, lr=5e-5, patience=5, ckpt_name='approach_A')

history_A = pd.concat([hist_A1, hist_A2], ignore_index=True)
print('Approach A final:', {k:v for k,v in metrics_A.items() if k!='conf_matrix'})


---
## Approach B — Pose + BiLSTM
Replicates Fang, Yeung & Fujii (2024).


### Cell B1 — MediaPipe pose extraction

In [ ]:
import mediapipe as mp
mp_pose    = mp.solutions.pose
N_KP       = 33
N_FRAMES_B = 16


def extract_pose_sequence(video_path, n_frames=N_FRAMES_B):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened(): return None
    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, max(total-1,0), n_frames, dtype=int)
    seq     = []
    with mp_pose.Pose(static_image_mode=True, model_complexity=1,
                      min_detection_confidence=0.3) as pose:
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, frame = cap.read()
            if not ok: seq.append(np.zeros(N_KP*3)); continue
            result = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if result.pose_landmarks:
                kps = [[lm.x,lm.y,lm.visibility] for lm in result.pose_landmarks.landmark]
            else:
                kps = [[0.0,0.0,0.0]]*N_KP
            seq.append(np.array(kps).flatten())
    cap.release()
    return np.array(seq, dtype=np.float32)


def get_pose_cached(action_id, video_paths, n_frames=N_FRAMES_B):
    cache = CACHE_DIR / f'pose_{action_id}.npy'
    if cache.exists(): return np.load(cache)
    replays = [v for v in video_paths if 'replay' in str(v).lower()]
    for vp in (replays or video_paths):
        seq = extract_pose_sequence(vp, n_frames)
        if seq is not None:
            np.save(cache, seq); return seq
    seq = np.zeros((n_frames, N_KP*3), dtype=np.float32)
    np.save(cache, seq); return seq


if HAS_VIDEOS:
    all_ids = list(train_df['action_id']) + list(valid_df['action_id'])
    already = {f.stem.replace('pose_','') for f in CACHE_DIR.glob('pose_*.npy')}
    to_do   = [a for a in all_ids if a not in already]
    print(f'Pose cache: {len(already)} ready, {len(to_do)} to extract')
    for i, aid in enumerate(to_do):
        get_pose_cached(aid, video_lookup.get(aid,[]))
        if (i+1) % 20 == 0: print(f'  {i+1}/{len(to_do)}')
    print('Pose extraction complete')
else:
    print('Synthetic mode')


### Cell B2 — BiLSTM model and training

In [ ]:
class MVFoulPoseDataset(Dataset):
    def __init__(self, df, video_lookup, n_frames=N_FRAMES_B):
        self.df=df.reset_index(drop=True); self.vl=video_lookup
        self.n_frames=n_frames; self.synthetic=len(video_lookup)==0
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]; label=int(row['label'])
        if self.synthetic:
            return torch.randn(self.n_frames, N_KP*3), label
        clips = self.vl.get(row['action_id'],[])
        seq   = get_pose_cached(row['action_id'], clips, self.n_frames)
        return torch.from_numpy(seq), label


class ApproachB_BiLSTM(nn.Module):
    def __init__(self, in_features=N_KP*3, hidden=256, n_layers=2,
                 n_classes=N_CLASSES, dropout=0.3):
        super().__init__()
        self.proj = nn.Linear(in_features, hidden)
        self.lstm = nn.LSTM(hidden, hidden, n_layers, batch_first=True,
                            bidirectional=True, dropout=dropout)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden*2, n_classes))
    def forward(self, x):
        x = F.relu(self.proj(x))
        out, _ = self.lstm(x)
        return self.head(out.mean(1))

train_dl_B = DataLoader(MVFoulPoseDataset(train_df, video_lookup),
                        batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
valid_dl_B = DataLoader(MVFoulPoseDataset(valid_df, video_lookup),
                        batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

model_B = ApproachB_BiLSTM().to(DEVICE)
print(f'Approach B | params: {sum(p.numel() for p in model_B.parameters()):,}')

metrics_B, history_B = train_approach('B-BiLSTM', model_B, train_dl_B, valid_dl_B,
    n_epochs=30, lr=3e-4, patience=7, ckpt_name='approach_B')
print('Approach B final:', {k:v for k,v in metrics_B.items() if k!='conf_matrix'})


---
## Approach C — ST-GCN (Novel Contribution)
Velocity-augmented skeleton graphs. Node features: [x, y, vx, vy].
Pure PyTorch implementation — no external graph library.


### Cell C1 — Skeleton graph + feature extraction

In [ ]:
COCO_FROM_MP = [0,2,5,7,8,11,12,13,14,15,16,23,24,25,26,27,28]
N_JOINTS = 17
JOINT_NAMES = ['Nose','L-eye','R-eye','L-ear','R-ear',
               'L-shoulder','R-shoulder','L-elbow','R-elbow',
               'L-wrist','R-wrist','L-hip','R-hip',
               'L-knee','R-knee','L-ankle','R-ankle']
SKELETON_EDGES = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16),
]


def build_adjacency(n_joints, edges):
    A = np.zeros((n_joints,n_joints), dtype=np.float32)
    for i,j in edges: A[i,j]=A[j,i]=1.0
    A += np.eye(n_joints, dtype=np.float32)
    D = np.diag(A.sum(axis=1)**-0.5)
    return torch.tensor(D@A@D, dtype=torch.float32)

A_HAT = build_adjacency(N_JOINTS, SKELETON_EDGES)
print(f'A_HAT: {A_HAT.shape}  edges: {len(SKELETON_EDGES)}')


def pose_to_graph(seq):
    T    = seq.shape[0]
    full = seq.reshape(T, 33, 3)
    coco = full[:, COCO_FROM_MP, :2]  # (T,17,2)
    vel  = np.zeros_like(coco)
    vel[1:] = coco[1:] - coco[:-1]
    return np.concatenate([coco, vel], axis=-1).astype(np.float32)  # (T,17,4)


def get_graph_cached(action_id, video_paths, n_frames=N_FRAMES_B):
    cache = CACHE_DIR / f'graph_{action_id}.npy'
    if cache.exists(): return np.load(cache)
    pose_f = CACHE_DIR / f'pose_{action_id}.npy'
    if pose_f.exists():
        seq = np.load(pose_f)
    else:
        replays = [v for v in video_paths if 'replay' in str(v).lower()]
        seq = None
        for vp in (replays or video_paths):
            seq = extract_pose_sequence(vp, n_frames)
            if seq is not None: break
        if seq is None:
            seq = np.zeros((n_frames, 33*3), dtype=np.float32)
    feats = pose_to_graph(seq)
    np.save(cache, feats); return feats


if HAS_VIDEOS:
    all_ids = list(train_df['action_id']) + list(valid_df['action_id'])
    already = {f.stem.replace('graph_','') for f in CACHE_DIR.glob('graph_*.npy')}
    to_do   = [a for a in all_ids if a not in already]
    print(f'Graph cache: {len(already)} ready, {len(to_do)} to extract')
    for i, aid in enumerate(to_do):
        get_graph_cached(aid, video_lookup.get(aid,[]))
        if (i+1) % 50 == 0: print(f'  {i+1}/{len(to_do)}')
    print('Graph extraction complete')
else:
    print('Synthetic mode')


### Cell C2 — ST-GCN model

In [ ]:
class SpatialGCN(nn.Module):
    def __init__(self, c_in, c_out, A):
        super().__init__()
        self.register_buffer('A', A)
        self.W  = nn.Linear(c_in, c_out, bias=False)
        self.bn = nn.BatchNorm2d(c_out)
    def forward(self, x):
        B,C,T,N = x.shape
        h = x.permute(0,2,3,1).reshape(B*T,N,C)
        h = self.W(h)
        h = torch.bmm(self.A.unsqueeze(0).expand(B*T,-1,-1), h)
        h = h.reshape(B,T,N,-1).permute(0,3,1,2)
        return F.relu(self.bn(h))


class STGCNBlock(nn.Module):
    def __init__(self, c_in, c_out, A, t_kernel=9, stride=1, dropout=0.2):
        super().__init__()
        self.gcn = SpatialGCN(c_in, c_out, A)
        self.tcn = nn.Sequential(
            nn.Conv2d(c_out, c_out, (t_kernel,1), (stride,1), (t_kernel//2,0)),
            nn.BatchNorm2d(c_out), nn.ReLU(), nn.Dropout2d(dropout))
        self.res = (nn.Sequential(nn.Conv2d(c_in,c_out,1,(stride,1)), nn.BatchNorm2d(c_out))
                    if c_in != c_out or stride != 1 else nn.Identity())
    def forward(self, x):
        return F.relu(self.tcn(self.gcn(x)) + self.res(x))


class ApproachC_STGCN(nn.Module):
    def __init__(self, c_in=4, n_classes=N_CLASSES, A=None, dropout=0.2):
        super().__init__()
        A = A if A is not None else A_HAT
        self.bn_in  = nn.BatchNorm1d(c_in*N_JOINTS)
        self.layers = nn.ModuleList([
            STGCNBlock(c_in, 64,  A, dropout=dropout),
            STGCNBlock(64,   128, A, dropout=dropout),
            STGCNBlock(128,  256, A, dropout=dropout),
        ])
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(dropout), nn.Linear(256, n_classes))
    def forward(self, x):
        B,C,T,N = x.shape
        xn = x.permute(0,2,1,3).reshape(B*T,C*N)
        xn = self.bn_in(xn).reshape(B,T,C,N).permute(0,2,1,3)
        for layer in self.layers: xn = layer(xn)
        return self.head(xn)

# Sanity check
_d = torch.randn(2,4,N_FRAMES_B,N_JOINTS).to(DEVICE)
_m = ApproachC_STGCN(A=A_HAT.to(DEVICE)).to(DEVICE)
_o = _m(_d)
print(f'ST-GCN: {tuple(_d.shape)} -> {tuple(_o.shape)}')
print(f'Approach C | params: {sum(p.numel() for p in _m.parameters()):,}')
del _d, _m, _o; torch.cuda.empty_cache()


### Cell C3 — ST-GCN training

In [ ]:
class MVFoulGraphDataset(Dataset):
    def __init__(self, df, video_lookup, n_frames=N_FRAMES_B):
        self.df=df.reset_index(drop=True); self.vl=video_lookup
        self.n_frames=n_frames; self.synthetic=len(video_lookup)==0
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]; label=int(row['label'])
        if self.synthetic:
            feats = np.random.randn(self.n_frames, N_JOINTS, 4).astype(np.float32)
        else:
            clips = self.vl.get(row['action_id'],[])
            feats = get_graph_cached(row['action_id'], clips, self.n_frames)
        return torch.from_numpy(feats).permute(2,0,1), label  # (4,T,N)

train_dl_C = DataLoader(MVFoulGraphDataset(train_df, video_lookup),
                        batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
valid_dl_C = DataLoader(MVFoulGraphDataset(valid_df, video_lookup),
                        batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

model_C = ApproachC_STGCN(A=A_HAT.to(DEVICE)).to(DEVICE)

metrics_C, history_C = train_approach('C-STGCN', model_C, train_dl_C, valid_dl_C,
    n_epochs=40, lr=1e-3, patience=8, ckpt_name='approach_C')
print('Approach C final:', {k:v for k,v in metrics_C.items() if k!='conf_matrix'})


---
## Comparison — unified results

In [ ]:
all_results = {
    'A - MViT (video)'  : metrics_A,
    'B - BiLSTM (pose)' : metrics_B,
    'C - ST-GCN (novel)': metrics_C,
}
results_df = pd.DataFrame([
    {'Approach':name,'Balanced Acc':m['balanced_acc'],
     'Dive F1':m['dive_f1'],'Macro F1':m['macro_f1']}
    for name, m in all_results.items()
]).set_index('Approach')

print('='*65)
print('FINAL COMPARISON -- Validation Set')
print('='*65)
print(results_df.to_string(float_format='{:.4f}'.format))
if not HAS_VIDEOS:
    print('SYNTHETIC MODE -- numbers are random, re-run with real clips')

results_df.to_csv(RESULTS_DIR / 'comparison_table.csv')
print(f'Saved: {RESULTS_DIR}/comparison_table.csv')


In [ ]:
# Learning curves
fig, axes = plt.subplots(1,3,figsize=(15,4),sharey=True)
for ax, (name, hist, col) in zip(axes, [
    ('A MViT',  history_A, '#378ADD'),
    ('B BiLSTM',history_B, '#1D9E75'),
    ('C STGCN', history_C, '#E24B4A')]):
    ax.plot(hist['epoch'],hist['tr_ba'],color=col,alpha=0.4,linestyle='--',label='Train BA')
    ax.plot(hist['epoch'],hist['val_ba'],color=col,lw=2,label='Valid BA')
    ax.plot(hist['epoch'],hist['val_dive_f1'],color=col,lw=1.5,ls=':',label='Dive F1')
    ax.set_title(name); ax.set_xlabel('Epoch'); ax.set_ylim(0,1)
    ax.legend(fontsize=8,frameon=False)
axes[0].set_ylabel('Score')
plt.suptitle('Learning curves',y=1.02); plt.tight_layout()
plt.savefig(RESULTS_DIR/'learning_curves.png',bbox_inches='tight'); plt.show()

# Confusion matrices
fig, axes = plt.subplots(1,3,figsize=(18,5))
for ax,(name,m) in zip(axes,all_results.items()):
    plot_confusion_matrix(m['conf_matrix'],name,ax)
plt.suptitle('Confusion matrices (row-normalised)',y=1.02); plt.tight_layout()
plt.savefig(RESULTS_DIR/'confusion_matrices.png',bbox_inches='tight'); plt.show()

# Bar comparison
fig, ax = plt.subplots(figsize=(9,4))
x, w = np.arange(len(results_df)), 0.25
b1=ax.bar(x-w,results_df['Balanced Acc'],w,label='Balanced Acc',color='#378ADD',alpha=0.85)
b2=ax.bar(x,  results_df['Dive F1'],     w,label='Dive F1',     color='#E24B4A',alpha=0.85)
b3=ax.bar(x+w,results_df['Macro F1'],    w,label='Macro F1',    color='#1D9E75',alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(results_df.index,fontsize=9)
ax.set_ylim(0,1.1); ax.set_ylabel('Score')
ax.set_title('Approach comparison -- validation set'); ax.legend(frameon=False)
for bars in (b1,b2,b3): ax.bar_label(bars,fmt='%.3f',padding=2,fontsize=7)
plt.tight_layout()
plt.savefig(RESULTS_DIR/'bar_comparison.png',bbox_inches='tight'); plt.show()


### ST-GCN joint importance

In [ ]:
@torch.no_grad()
def joint_importance(model, valid_dl, joint_idx):
    model.eval()
    preds, trues = [], []
    for x, y in valid_dl:
        xc = x.clone().to(DEVICE)
        xc[:,:,:,joint_idx] = 0.0
        with autocast(): preds.extend(model(xc).argmax(1).cpu().numpy())
        trues.extend(y.numpy())
    return f1_score(trues, preds, labels=[DIVE_IDX], average='macro', zero_division=0)

print('Joint importance ablation...')
baseline_f1 = metrics_C['dive_f1']
importance  = []
for j, jname in enumerate(JOINT_NAMES):
    drop = baseline_f1 - joint_importance(model_C, valid_dl_C, j)
    importance.append({'joint':jname,'dive_f1_drop':drop})
    print(f'  {jname:<14s} drop: {drop:+.4f}')

imp_df = pd.DataFrame(importance).sort_values('dive_f1_drop',ascending=False)
fig, ax = plt.subplots(figsize=(9,5))
colors  = ['#E24B4A' if d>0 else '#9FE1CB' for d in imp_df['dive_f1_drop']]
ax.barh(imp_df['joint'][::-1],imp_df['dive_f1_drop'][::-1],
        color=colors[::-1],edgecolor='white')
ax.axvline(0,color='gray',lw=0.8)
ax.set_xlabel('Dive F1 drop when joint zeroed')
ax.set_title('ST-GCN joint importance for Dive classification\n'
             '(contact-response: ankles/knees expected most diagnostic)')
plt.tight_layout()
plt.savefig(RESULTS_DIR/'joint_importance.png',bbox_inches='tight'); plt.show()
imp_df.to_csv(RESULTS_DIR/'joint_importance.csv',index=False)
print('Top 3:', imp_df.head(3)[['joint','dive_f1_drop']].to_string(index=False))


### Save artefacts

In [ ]:
for nm in ['approach_A','approach_B','approach_C']:
    p = CKPT_DIR / f'{nm}.pt'
    size = f'{p.stat().st_size/1e6:.1f} MB' if p.exists() else 'MISSING'
    print(f'  {nm}.pt: {size}')

summary = {}
for nm, m in all_results.items():
    summary[nm] = {k: float(v) if not isinstance(v,np.ndarray) else v.tolist()
                  for k,v in m.items()}
with open(RESULTS_DIR/'full_results.json','w') as f:
    json.dump(summary, f, indent=2)

print(f'All outputs in {RESULTS_DIR}:')
for p in sorted(RESULTS_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size/1e3:.1f} KB)')
